# Japan CDM ETL

한국 `codeToRun_2.R` 과 같이 **아래 셀에서 TRUE/FALSE 를 바꾼 뒤** `ETL 실행` 셀만 반복 실행하면 됩니다.

CLI 는 프로젝트 루트의 `codeToRun_japan.py` 상단 변수를 동일하게 수정합니다.

## 0) 경로 및 설정

In [1]:
import os
import sys
import subprocess

ROOT = os.getcwd()
if not os.path.exists(os.path.join(ROOT, "codeToRun_japan.ipynb")):
    ROOT = os.path.dirname(ROOT)
SCRIPTS = os.path.join(ROOT, "ETL---Japan-Cohort", "etlJapanCohort", "scripts")

def run(script_name):
    path = os.path.join(SCRIPTS, script_name)
    if not os.path.exists(path):
        print(f"[ERROR] Not found: {path}")
        return 1
    return subprocess.call([sys.executable, path], cwd=ROOT)

print("ROOT:", ROOT)
print("SCRIPTS:", SCRIPTS)

ROOT: c:\Users\chaeyoon\Desktop\koreajapan
SCRIPTS: c:\Users\chaeyoon\Desktop\koreajapan\ETL---Japan-Cohort\etlJapanCohort\scripts


## 실행 단계 제어 (TRUE/FALSE)
처음에는 `CDM_ddl = True` 로 빈 CDM 테이블을 만듭니다.

In [2]:
CDM_ddl = False
create_database_if_missing = True
master_table = False
import_vocabulary_copy = False
import_vocabulary_bulk = False

## 데이터 적재 (필요한 부분만 TRUE)

**실행 순서**는 아래 변수를 적은 줄 순서가 아니라, `phase1_setup.ETL_STEP_SQL`에 고정됨:  
`person` → `death` → `observation_period` → `visit_occurrence` → `condition_occurrence` → `drug_exposure` → `procedure_occurrence`  
여러 개를 동시에 `True`로 두어도 항상 이 순서로만 실행됨.

In [3]:
# --- Phase1 도메인 (ETL_STEP_SQL 실행 순서와 동일하게 배치) ---
person = False
death = False
observation_period = False
visit_occurrence = False
condition_occurrence = False
drug_exposure = False
procedure_occurrence = True

# --- Procedure 로컬 매핑 파이프라인(실험) ---
# procedure_occurrence 실행 전에 로컬 매핑 테이블을 자동 생성/갱신.
# - None: 매핑 파이프라인 스킵
# - "rules_only": 내부 규칙(마스터 테이블/ICD9CM 등)만
# - "rules_plus_external": 규칙 + 외부 매핑 도구 CSV(점수 임계값 이상 자동 반영)
procedure_mapping_mode = "rules_only"  # None | "rules_only" | "rules_plus_external"
procedure_mapping_auto_score_threshold = 0.90
procedure_external_mapping_csv = None  # 예: r"E:\...\\procedure_mapping_results.csv"
procedure_mapping_export_csv = None   # JP_PROCEDURE_MASTER 참조용 CSV (임의 도구 입력)

# --- 미구현: 켜도 [TODO] 로그만 (일본용 SQL 없음) ---
location = False
care_site = False
observation = False
device_exposure = False
measurement = False
payer_plan_period = False
cost = False

## 후처리

In [4]:
generateEra = False
dose_era = False
cdm_source = False
indexing = False
constraints = False
data_cleansing = False

## ETL 실행
위 플래그로 `run_japan_etl` 호출. 용어 적재 플래그가 True 이면 그에 맞는 스크립트를 먼저 실행합니다.

`execute_japan_etl` / `phase1_setup` 를 디스크에서 고쳤다면, 아래 셀이 `importlib.reload` 로 **항상 최신 모듈**을 쓰도록 되어 있음 (진행 로그 등 반영).

**후처리** 셀을 실행하지 않아도 아래 셀에서 플래그 기본값을 쓰므로 `NameError` 나지 않음.

In [ ]:
import importlib

sys.path.insert(0, SCRIPTS)
# 스크립트 수정 후에도 최신 코드 사용 (커널 캐시 우회)
import execute_japan_etl
importlib.reload(execute_japan_etl)
from execute_japan_etl import run_japan_etl

# 위 셀을 건너뛰어도 NameError 방지 (미정의 시 기본값)
_g = globals()
def _F(name, default=False):
    if name == "create_database_if_missing":
        return _g.get(name, True)
    return _g.get(name, default)

if _F("import_vocabulary_copy"):
    run("import_vocabulary_copy.py")
if _F("import_vocabulary_bulk"):
    run("import_vocabulary.py")

flags = {
    "CDM_ddl": _F("CDM_ddl"),
    "create_database_if_missing": _F("create_database_if_missing", True),
    "master_table": _F("master_table"),
    "person": _F("person"),
    "death": _F("death"),
    "observation_period": _F("observation_period"),
    "visit_occurrence": _F("visit_occurrence"),
    "condition_occurrence": _F("condition_occurrence"),
    "drug_exposure": _F("drug_exposure"),
    "procedure_occurrence": _F("procedure_occurrence"),

    # --- Procedure 로컬 매핑 파이프라인(실험) ---
    "procedure_mapping_mode": _F("procedure_mapping_mode"),
    "procedure_mapping_auto_score_threshold": _F("procedure_mapping_auto_score_threshold", 0.90),
    "procedure_external_mapping_csv": _F("procedure_external_mapping_csv"),
    "procedure_mapping_export_csv": _F("procedure_mapping_export_csv"),

    "location": _F("location"),
    "care_site": _F("care_site"),
    "observation": _F("observation"),
    "device_exposure": _F("device_exposure"),
    "measurement": _F("measurement"),
    "payer_plan_period": _F("payer_plan_period"),
    "cost": _F("cost"),
    "generateEra": _F("generateEra"),
    "dose_era": _F("dose_era"),
    "cdm_source": _F("cdm_source"),
    "indexing": _F("indexing"),
    "constraints": _F("constraints"),
    "data_cleansing": _F("data_cleansing"),
}
run_japan_etl(flags)


=== run_japan_etl ===

[Phase0] Database 'japan_cohort_cdm' already exists.
[Phase0] CDM_ddl=False - skipping 000 DDL.
[Domain ETL]
  Domain steps enabled: 1  →  procedure_occurrence

  --- Step 1/1: procedure_occurrence ---
  [procedure_mapping] mode=rules_only  thr=0.9


c:\Users\chaeyoon\Desktop\koreajapan\ETL---Japan-Cohort\etlJapanCohort\scripts\phase0_setup.py:51: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine_master.connect() as conn:


  Running 110.Procedure_occurrence_japan.sql  (2 SQL batches)...
    [1/2] /**************************************
    [1/2] ok  (0.1s)
    [2/2] INSERT INTO japan_cohort_cdm.dbo.procedure_occurrence (


c:\Users\chaeyoon\Desktop\koreajapan\ETL---Japan-Cohort\etlJapanCohort\scripts\phase1_setup.py:84: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


## (선택) 원스텝 스크립트만 실행

In [ ]:
# run("load_sas_to_sql_japan.py")
# run("phase0_setup.py")
# run("import_vocabulary_copy.py")
# run("import_vocabulary.py")
pass